# One-time preference data, then DMPO → DMPO+DEPO trials

This notebook is preset for the completed 30-task `swesmith-pilot-20260719` trajectory collection. It submits exactly two one-time CPU data jobs (one DMPO pair builder and one DEPO trajectory builder), then keeps all later submissions training-only. The first training phase is baseline SFT → DMPO. After inspecting that model, the second phase reuses the selected DMPO package and trains DMPO → DEPO. All switches that submit jobs default to `False`.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').is_file(), ROOT

# Change only RUN_NAME when moving from this 30-task pilot to the 5K collection.
RUN_NAME = os.getenv('RUN_NAME', 'swesmith-pilot-20260719')
scratch = Path(os.getenv('DEBUG_DEPO_SCRATCH', ROOT / 'scratch' / 'cluster-artifacts'))
RUN_ROOT = Path(os.getenv('RUN_ROOT', scratch / 'runs' / RUN_NAME)).expanduser().resolve()
BASE_MODEL = os.getenv('BASE_MODEL', 'Kwai-Klear/Klear-AgentForge-8B-SFT')

# Small pilot: deterministic subsets of the immutable data and shorter context.
MAX_TRAIN_ROWS = os.getenv('MAX_TRAIN_ROWS', '64')
MAX_LENGTH = os.getenv('TRAIN_MAX_LENGTH', '8192')
EPOCHS = os.getenv('EPOCHS', '1')
GRADIENT_ACCUMULATION_STEPS = os.getenv('GRADIENT_ACCUMULATION_STEPS', '8')
SAVE_STEPS = os.getenv('SAVE_STEPS', '2')
PREFERENCE_MAX_ROLLOUTS = os.getenv('PREFERENCE_MAX_ROLLOUTS', '4')

DMPO_TRIAL_NAME = os.getenv('DMPO_TRIAL_NAME', 'pilot-64-lr1e6-g07')
DEPO_TRIAL_NAME = os.getenv('DEPO_TRIAL_NAME', 'pilot-64-lr2e5-a2')
DMPO_LEARNING_RATE = os.getenv('DMPO_LEARNING_RATE', '1e-6')
DMPO_BETA = os.getenv('DMPO_BETA', '0.1')
DMPO_GAMMA = os.getenv('DMPO_GAMMA', '0.7')
DEPO_LEARNING_RATE = os.getenv('DEPO_LEARNING_RATE', '2e-5')
DEPO_BETA = os.getenv('DEPO_BETA', '0.2')
ALPHA_TOKENS = os.getenv('ALPHA_TOKENS', '2.0')
ALPHA_STEPS = os.getenv('ALPHA_STEPS', '2.0')

# Five previously exercised SWE-bench Verified tasks keep model evaluation cheap.
EVAL_TASK_IDS_FILE = ROOT / 'data/splits/swebench_verified_pilot_5_instance_ids.txt'
EVAL_EXPECTED_TASKS = '5'
EVAL_NUM_SHARDS = '1'
EVAL_ROLLOUT_WORKERS = '2'
EVAL_CONTEXT_LENGTH = '8192'

common_env = os.environ.copy()
common_env.update({
    'RUN_NAME': RUN_NAME,
    'RUN_ROOT': str(RUN_ROOT),
    'BASE_MODEL': BASE_MODEL,
    'PREFERENCE_MAX_ROLLOUTS': PREFERENCE_MAX_ROLLOUTS,
    'PREFERENCE_DATA_MODE': 'reuse',
    'NUM_PROCESSES': '1',
    'MAX_TRAIN_ROWS': MAX_TRAIN_ROWS,
    'MAX_LENGTH': MAX_LENGTH,
    'EPOCHS': EPOCHS,
    'GRADIENT_ACCUMULATION_STEPS': GRADIENT_ACCUMULATION_STEPS,
    'SAVE_STEPS': SAVE_STEPS,
    'DMPO_TRIAL_NAME': DMPO_TRIAL_NAME,
    'DEPO_TRIAL_NAME': DEPO_TRIAL_NAME,
    'DMPO_LEARNING_RATE': DMPO_LEARNING_RATE,
    'DMPO_BETA': DMPO_BETA,
    'DMPO_GAMMA': DMPO_GAMMA,
    'DEPO_LEARNING_RATE': DEPO_LEARNING_RATE,
    'DEPO_BETA': DEPO_BETA,
    'ALPHA_TOKENS': ALPHA_TOKENS,
    'ALPHA_STEPS': ALPHA_STEPS,
    'EVAL_TASK_IDS_FILE': str(EVAL_TASK_IDS_FILE),
    'EVAL_EXPECTED_TASKS': EVAL_EXPECTED_TASKS,
    'EVAL_NUM_SHARDS': EVAL_NUM_SHARDS,
    'EVAL_ROLLOUT_WORKERS': EVAL_ROLLOUT_WORKERS,
    'EVAL_CONTEXT_LENGTH': EVAL_CONTEXT_LENGTH,
})
DMPO_MODEL = RUN_ROOT / 'experiments' / 'dmpo' / DMPO_TRIAL_NAME / 'model'
DEPO_MODEL = RUN_ROOT / 'experiments' / 'dmpo-depo' / DMPO_TRIAL_NAME / 'depo' / DEPO_TRIAL_NAME / 'model'
print(json.dumps({'run_root': str(RUN_ROOT), 'dmpo_model': str(DMPO_MODEL), 'depo_model': str(DEPO_MODEL)}, indent=2))

## 1. Confirm the trajectory collection

The current pilot should report three shards, 30 tasks, eight rollout slots, and complete evaluation summaries. This cell is read-only.

In [ ]:
from debug_depo.preference_data import parse_sample_indices, select_sample_indices
from debug_depo.swesmith_progress import inspect_collection

progress = inspect_collection(RUN_ROOT)
collection_problems = list(progress.warnings)
collection_problems.extend(
    f'shard-{shard.index}: {shard.state}'
    for shard in progress.shards
    if shard.state != 'complete'
)
assert not collection_problems, collection_problems
expected_ids = progress.expected_task_ids
assert expected_ids, f'No expected task IDs found under {RUN_ROOT}'

sample_spec = common_env.get('PREFERENCE_SAMPLE_INDICES', '').strip()
selected_samples = select_sample_indices(
    RUN_ROOT,
    max_rollouts=int(PREFERENCE_MAX_ROLLOUTS),
    sample_indices=parse_sample_indices(sample_spec) if sample_spec else None,
)
evaluation_problems = []
for sample_index in selected_samples:
    summary_path = RUN_ROOT / 'evaluation' / f'sample-{sample_index}' / 'summary.json'
    if not summary_path.is_file():
        evaluation_problems.append(f'sample-{sample_index}: missing {summary_path}')
        continue
    summary = json.loads(summary_path.read_text())
    result_ids = {
        str(result.get('instance_id'))
        for result in summary.get('results', [])
        if isinstance(result, dict)
    }
    if summary.get('scored_instances') != len(expected_ids) or result_ids != expected_ids:
        evaluation_problems.append(
            f"sample-{sample_index}: scored={summary.get('scored_instances')}, "
            f'IDs={len(result_ids)}, expected={len(expected_ids)}'
        )
assert not evaluation_problems, evaluation_problems
preference_data_ready = True
print(json.dumps({
    'shards': len(progress.shards),
    'expected_tasks': len(expected_ids),
    'collected_tasks': progress.collected_tasks,
    'rollouts_per_task': progress.samples_per_task,
    'selected_evaluated_samples': selected_samples,
    'preference_data_ready': preference_data_ready,
}, indent=2))

## 2. Submit the two data jobs once

These are independent CPU jobs and run in parallel. Each atomically creates a hash-validated immutable artifact. Re-running the cell reuses complete artifacts; it does not rebuild them. Set `REBUILD_PREFERENCE_DATA=1` only if you intentionally want to replace both datasets for this collection.

In [ ]:
PREVIEW_DATA_JOBS = True
SUBMIT_DATA_JOBS = False
REBUILD_PREFERENCE_DATA = False
data_env = common_env | {'REBUILD_PREFERENCE_DATA': '1' if REBUILD_PREFERENCE_DATA else '0'}
if PREVIEW_DATA_JOBS:
    subprocess.run(['bash', 'cluster/submit_preference_data.sh'], cwd=ROOT, env=data_env | {'DRY_RUN': '1'}, check=True)
if SUBMIT_DATA_JOBS:
    assert globals().get('preference_data_ready') is True, 'Run the collection/evaluation gate first'
    assert shutil.which('qsub'), 'Run this cell on the cluster login node'
    subprocess.run(['bash', 'cluster/submit_preference_data.sh'], cwd=ROOT, env=data_env, check=True)

## 3. Validate the immutable datasets

Run this after both data jobs finish. Training submissions perform the same validation automatically.

In [ ]:
for objective in ('dmpo', 'depo'):
    subprocess.run(['bash', 'scripts/validate_preference_data.sh', objective], cwd=ROOT, env=common_env, check=True)

## 4. Train and evaluate DMPO first

This submission does not rebuild preference data. It trains the named DMPO trial, packages it on CPU, then evaluates it on the five-task pilot split. Re-submit the same trial after a cluster failure to resume it. Change `DMPO_TRIAL_NAME` whenever you change a hyperparameter.

In [ ]:
PREVIEW_DMPO = True
SUBMIT_DMPO = False
dmpo_env = common_env | {
    'EXPERIMENT_ARM': 'dmpo',
    'DMPO_EVAL_RUN_NAME': f'{RUN_NAME}-dmpo-{DMPO_TRIAL_NAME}-pilot-5',
}
if PREVIEW_DMPO:
    subprocess.run(['bash', 'cluster/submit_preference_training.sh'], cwd=ROOT, env=dmpo_env | {'DRY_RUN': '1'}, check=True)
if SUBMIT_DMPO:
    assert shutil.which('qsub'), 'Run this cell on the cluster login node'
    subprocess.run(['bash', 'cluster/submit_preference_training.sh'], cwd=ROOT, env=dmpo_env, check=True)

## 5. Inspect DMPO before continuing

After pulling cluster artifacts, this confirms the standalone model and prints any available five-task analysis summary. Keep `SUBMIT_DEPO=False` until you want to continue from this DMPO trial.

In [ ]:
print('DMPO package complete:', (DMPO_MODEL / 'package_manifest.json').is_file())
dmpo_eval_root = RUN_ROOT.parent / f'{RUN_NAME}-dmpo-{DMPO_TRIAL_NAME}-pilot-5'
for candidate in (dmpo_eval_root / 'analysis' / 'summary.json', dmpo_eval_root / 'analysis-full' / 'summary.json'):
    if candidate.is_file():
        print(candidate)
        print(json.dumps(json.loads(candidate.read_text()), indent=2)[:8000])
        break
else:
    print('DMPO pilot analysis has not been pulled yet:', dmpo_eval_root)

## 6. Train DMPO → DEPO after selecting DMPO

`DMPO_MODE=reuse` makes this a DEPO-only training/package/evaluation submission. It validates and reuses both one-time datasets and initializes DEPO from the packaged DMPO model. It does not repeat DMPO evaluation.

In [ ]:
PREVIEW_DEPO = True
SUBMIT_DEPO = False
depo_env = common_env | {
    'EXPERIMENT_ARM': 'dmpo-depo',
    'DMPO_MODE': 'reuse',
    'SUBMIT_DMPO_EVALUATION': '0',
    'DEPO_EVAL_RUN_NAME': f'{RUN_NAME}-dmpo-{DMPO_TRIAL_NAME}-depo-{DEPO_TRIAL_NAME}-pilot-5',
}
if PREVIEW_DEPO:
    subprocess.run(['bash', 'cluster/submit_preference_training.sh'], cwd=ROOT, env=depo_env | {'DRY_RUN': '1'}, check=True)
if SUBMIT_DEPO:
    assert shutil.which('qsub'), 'Run this cell on the cluster login node'
    assert (DMPO_MODEL / 'package_manifest.json').is_file(), f'Missing selected DMPO package: {DMPO_MODEL}'
    subprocess.run(['bash', 'cluster/submit_preference_training.sh'], cwd=ROOT, env=depo_env, check=True)

## 7. Repeat training without rebuilding data

For another DMPO configuration, change `DMPO_TRIAL_NAME` plus the DMPO hyperparameters and rerun the DMPO submission. For another DEPO configuration on the selected DMPO model, keep `DMPO_TRIAL_NAME`, change `DEPO_TRIAL_NAME` plus DEPO hyperparameters, and rerun only the DEPO submission. Trial configuration and the data SHA-256 are locked in `trial_config.json`; incompatible reuse is rejected. Completed stages are no-ops, interrupted checkpoints resume, and interrupted packages are preserved before an atomic rebuild.

For the 5K collection, change `RUN_NAME`, submit the two data jobs once, then set `MAX_TRAIN_ROWS=0`, restore the desired context/epochs, and repeat the same DMPO-first → DEPO sequence.